In [2]:


from getpass import getpass

from langchain.agents import create_agent
from langchain_core.tools import tool

from langchain_google_genai import (
    ChatGoogleGenerativeAI,
    GoogleGenerativeAIEmbeddings
)

from langchain_community.vectorstores import FAISS
from langchain_text_splitters import CharacterTextSplitter
from langgraph.checkpoint.memory import InMemorySaver



GOOGLE_API_KEY = getpass("Enter Gemini API Key: ")



llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    temperature=0,
    google_api_key=GOOGLE_API_KEY
)



sample_text = """
LangChain is a framework for developing applications powered by
large language models.

LangChain was created by Harrison Chase.

LangChain provides tools for building AI applications using
language models, prompts, chains, agents, and retrieval augmented generation.

RAG stands for Retrieval Augmented Generation.
It retrieves relevant information from external documents
and uses that information to generate answers.

FAISS is a vector database library used for efficient similarity search.

Embeddings convert text into numerical vector representations.
These vectors help identify semantically similar documents.
"""

with open("sample.txt", "w", encoding="utf-8") as f:
    f.write(sample_text)

print("Sample file created successfully!")


with open("sample.txt", "r", encoding="utf-8") as f:
    text_data = f.read()



splitter = CharacterTextSplitter(
    separator="\n",
    chunk_size=300,
    chunk_overlap=50
)

texts = splitter.split_text(text_data)

print("Total chunks:", len(texts))


embedding = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    google_api_key=GOOGLE_API_KEY
)


vectorstore = FAISS.from_texts(
    texts=texts,
    embedding=embedding
)

retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)

print("FAISS Vector Database created successfully!")


@tool
def RAG_QA(question: str) -> str:
    """Use this tool to answer questions about LangChain."""

    # Retrieve relevant documents
    docs = retriever.invoke(question)

    if not docs:
        return "No relevant information found in the knowledge base."

    # Combine retrieved content
    context = "\n\n".join(
        doc.page_content for doc in docs
    )

    # Generate answer using Gemini
    prompt = f"""
    Answer the question using only the context below.

    If the answer is not available in the context,
    clearly state that it is not available.

    Context:
    {context}

    Question:
    {question}

    Answer:
    """

    answer = llm.invoke(prompt)

    return answer.content


checkpointer = InMemorySaver()

thread_config = {
    "configurable": {
        "thread_id": "rag-types-demo-1"
    }
}


agent = create_agent(
    model=llm,
    tools=[RAG_QA],
    system_prompt="""
    You are a helpful LangChain RAG assistant.

    Use the RAG_QA tool to answer questions about LangChain
    using the available knowledge base.

    Give clear and simple answers.
    Do not invent information.
    """,
    checkpointer=checkpointer
)

print("RAG Agent created successfully!")



print("\n1. First question")

res1 = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "What is LangChain?"
            }
        ]
    },
    config=thread_config
)

print("Answer:", res1["messages"][-1].content)



print("\n2. Follow-up")

res2 = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Who created it?"
            }
        ]
    },
    config=thread_config
)

print("Answer:", res2["messages"][-1].content)



print("\n3. Ask again")

res3 = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Explain LangChain again simply."
            }
        ]
    },
    config=thread_config
)

print("Answer:", res3["messages"][-1].content)



print("\n========== CHAT HISTORY ==========\n")

for message in res3["messages"]:
    if message.type in ["human", "ai"]:
        print(f"{message.type.upper()}: {message.content}\n")


Enter Gemini API Key: ··········
Sample file created successfully!
Total chunks: 3
FAISS Vector Database created successfully!
RAG Agent created successfully!

1. First question


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3935: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3935: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3935: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Answer: [{'type': 'text', 'text': 'LangChain is a framework for developing applications powered by large language models (LLMs).', 'extras': {'signature': 'EnsKeQFpFH0TtZckEt21l4XBbz4rV1NFEoD+AcMJ1/lYVOCvz2FDsHXN3u3dEehZR1dtKr8seNVCaFS4nlU5YyxIaqeGvktmj//GTsy1MeMF9jvCuWPL25TdMCKU0QjGMJ39b1dfn2EvJqXgGIQFC9qxGxZ0ch2yinUUDlQ='}}]

2. Follow-up


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3935: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3935: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3935: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Answer: [{'type': 'text', 'text': 'LangChain was created by Harrison Chase.', 'extras': {'signature': 'EnsKeQFpFH0TfeHmpt+QhjRF/BwGpqH3PHX9lLJ6eMCv4Hr/F5LQ5lIGiQqdnNVhoyZesVkKPq0aAe461luzxqNeMp9cmYH1LEJSl1bzx60GAFhkMBW2bckzzq/Ph41UzhGuVCeBmEG1I8Ik8w4o9RBD1nxk5U3sYoPmrjo='}}]

3. Ask again


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3935: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3935: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3935: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Answer: [{'type': 'text', 'text': "Think of LangChain as a helpful toolkit or bridge. It makes it easier to connect powerful AI language models (like GPT) to other data sources, tools, and systems so you can build complete, smart applications—like a chatbot that can search your company's documents or answer questions using real-time data.", 'extras': {'signature': 'EnsKeQFpFH0Tsf+WjMwTBriqNcv18UpE4rI1yBNi0TRCdHL6haR3n4bHtJg13IEos7Y7Q7ZNyQgjH0X4j9flwdRbxo6EPIgmj62+MVUYXyztl2UeOv3wZ0QL7y+LsEso43tqWTZ89LsBSDseU3dBJTNiROq9+RO5DQyCkUo='}}]

========== CHAT HISTORY ==========

HUMAN: What is LangChain?

AI: []

AI: [{'type': 'text', 'text': 'LangChain is a framework for developing applications powered by large language models (LLMs).', 'extras': {'signature': 'EnsKeQFpFH0TtZckEt21l4XBbz4rV1NFEoD+AcMJ1/lYVOCvz2FDsHXN3u3dEehZR1dtKr8seNVCaFS4nlU5YyxIaqeGvktmj//GTsy1MeMF9jvCuWPL25TdMCKU0QjGMJ39b1dfn2EvJqXgGIQFC9qxGxZ0ch2yinUUDlQ='}}]

HUMAN: Who created it?

AI: []

AI: [{'type': 'text', 'text':

In [2]:
%pip install -q --no-deps langchain-google-genai==4.4.3

In [6]:
%pip install -q filetype

In [7]:
%pip install -q langchain-community==0.4.2 faiss-cpu

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
